## Lista de Exercícios 17/04/2026 - Versão Pyomo

In [1]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory, TerminationCondition

In [23]:
# SOLVER_NAME exemplos: "gurobi", "cplex", "glpk", "cbc"
# SOLVER_CONFIGS = {}   exemplos: "TimeLimit" , "MIPGap"


def solve_model(model, tee=True, solver_name: str = "gurobi"):
    solver = SolverFactory(solver_name)
    if not solver.available(exception_flag=False):
        raise RuntimeError(
            f"Solver '{solver_name}' não está disponível para o Pyomo."
        )

    results = solver.solve(model, tee=tee)
    # results = solver.solve(model, tee=tee, options=SOLVER_CONFIGS)
    termination = results.solver.termination_condition
    if termination != TerminationCondition.optimal:
        print(f"Solução ótima não encontrada. Status: {termination}")
        return None

    return results

#### Ex1: Problema de Dimensionamento de Frota (Fleet Sizing Problem)

In [27]:
VEICULOS = ("A", "B")
CUSTO_COMBUSTIVEL = {"A": 110, "B": 75}

In [28]:
m = pyo.ConcreteModel(name="ex1-TransporteDeProdutos")

m.VEICULOS = pyo.Set(initialize=VEICULOS)
m.qtd_viagens_com = pyo.Var(m.VEICULOS, domain=pyo.NonNegativeIntegers)
m.custo_combustivel = pyo.Param(m.VEICULOS, initialize=CUSTO_COMBUSTIVEL)

m.obj = pyo.Objective(
    expr=pyo.summation(m.custo_combustivel, m.qtd_viagens_com),
    sense=pyo.minimize,
)

In [29]:
TIPOS = ("Refrigerado", "Não Refrigerado")
CAPACIDADE_VEICULO = {
    ("A", "Refrigerado"): 20,
    ("A", "Não Refrigerado"): 30,
    ("B", "Refrigerado"): 20,
    ("B", "Não Refrigerado"): 10,
}
DEMANDA_PRODUTO = {"Refrigerado": 160, "Não Refrigerado": 120}

In [30]:
def regra_capacidade(model, tipo):
    print("Aplicando regra para a restrição do tipo: ", tipo)

    soma = 0
    for v in model.VEICULOS:
        soma += model.capacidade_veiculo[v, tipo] * model.qtd_viagens_com[v]

    return soma >= model.demanda_produto[tipo]

In [ ]:
m.TIPOS = pyo.Set(initialize=TIPOS)
m.demanda_produto = pyo.Param(m.TIPOS, initialize=DEMANDA_PRODUTO)
m.capacidade_veiculo = pyo.Param(m.VEICULOS, m.TIPOS, initialize=CAPACIDADE_VEICULO)

m.capacidade = pyo.Constraint(
    m.TIPOS,
    rule=regra_capacidade
)

m.regra_B = pyo.Constraint(expr=m.qtd_viagens_com["B"] >= 4)

Aplicando regra para a restrição do tipo:  Refrigerado
Aplicando regra para a restrição do tipo:  Não Refrigerado


obs:
- m.capacidade é uma família de restrições
- ela é indexada por `m.TIPOS`
- para cada tipo em m.TIPOS, pyomo chama a função definida em `rule`

In [32]:
m.write("model1_PYOMO.lp", io_options={"symbolic_solver_labels": True})
results = solve_model(m)

Set parameter Username
Set parameter LicenseID to value 2810550
Academic license - for non-commercial use only - expires 2027-04-20
Read LP format model from file C:\Users\gratz\AppData\Local\Temp\tmpdbcec2na.pyomo.lp
Reading time = 0.00 seconds
x1: 3 rows, 2 columns, 5 nonzeros
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 2 columns and 5 nonzeros
Model fingerprint: 0x93960e82
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [8e+01, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e+00, 2e+02]
Found heuristic solution: objective 900.0000000
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros
Variable types: 0 

In [33]:
if results:
    print(f"\nMenor consumo de combustível: {pyo.value(m.obj)} L")
    for v in VEICULOS:
        print(
            f"Viagens com caminhão {v}: {int(round(pyo.value(m.qtd_viagens_com[v])))}"
        )


Menor consumo de combustível: 670.0 L
Viagens com caminhão A: 2
Viagens com caminhão B: 6


#### Ex3: Problema de Empacotamento (Packing Problem)

In [28]:
MATRIZES = [1, 2, 3, 4]

CUSTO_MATRIZ = {
    1: 1,
    2: 2,
    3: 3,
    4: 2,
}

TIPOS_CORTE = ("medio", "grande")

QTD_CORTE = {
    (1, "medio"): 8,
    (2, "medio"): 4,
    (3, "medio"): 2,
    (4, "medio"): 0,
    (1, "grande"): 0,
    (2, "grande"): 1,
    (3, "grande"): 2,
    (4, "grande"): 3,
}

DEMANDA = {
    "medio": 500,
    "grande": 350,
}

In [29]:
m = pyo.ConcreteModel(name="ex3-CorteDeChapasMetalicas")

m.MATRIZES = pyo.Set(initialize=MATRIZES)
m.matriz = pyo.Var(m.MATRIZES, domain=pyo.NonNegativeIntegers)

m.custo_matriz = pyo.Param(m.MATRIZES, initialize=CUSTO_MATRIZ)

m.TIPOS_CORTE = pyo.Set(initialize=TIPOS_CORTE)
m.qtd_corte = pyo.Param(m.MATRIZES, m.TIPOS_CORTE, initialize=QTD_CORTE)
m.demanda_corte = pyo.Param(m.TIPOS_CORTE, initialize=DEMANDA)

m.obj = pyo.Objective(
    expr=pyo.summation(m.custo_matriz, m.matriz),
    sense=pyo.minimize,
)

In [30]:
def regra_corte(model, tipo_corte):
    print("Aplicando regra para a restrição do tipo: ", tipo_corte)

    soma = 0
    for matriz in model.MATRIZES:
        soma += model.qtd_corte[matriz, tipo_corte] * model.matriz[matriz]

    return soma >= model.demanda_corte[tipo_corte]

In [31]:
m.corte = pyo.Constraint(m.TIPOS_CORTE, rule=regra_corte)

Aplicando regra para a restrição do tipo:  medio
Aplicando regra para a restrição do tipo:  grande


In [32]:
m.write("model3_PYOMO.lp", io_options={"symbolic_solver_labels": True})

('model3_PYOMO.lp', 2461682661360)

In [33]:
results = solve_model(m)

Set parameter Username
Set parameter LicenseID to value 2810550
Academic license - for non-commercial use only - expires 2027-04-20
Read LP format model from file C:\Users\gratz\AppData\Local\Temp\tmpqgfzw14c.pyomo.lp
Reading time = 0.00 seconds
x1: 2 rows, 4 columns, 6 nonzeros
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 4 columns and 6 nonzeros
Model fingerprint: 0xe67e1b6b
Variable types: 0 continuous, 4 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [1e+00, 3e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e+02, 5e+02]
Found heuristic solution: objective 750.0000000
Presolve time: 0.00s
Presolved: 2 rows, 4 columns, 6 nonzeros
Variable types: 0 continuous, 4 integer (0 binary)

Root

In [34]:
if results:
    print(f"\nCusto Total Mínimo: {pyo.value(m.obj)}\n")

    for i in MATRIZES:
        if pyo.value(m.matriz[i]) > 0:
            print(f"Matriz {i}: {int(round(pyo.value(m.matriz[i])))} chapas")

    medias = (
        8 * pyo.value(m.matriz[1])
        + 4 * pyo.value(m.matriz[2])
        + 2 * pyo.value(m.matriz[3])
    )
    grandes = (
        1 * pyo.value(m.matriz[2])
        + 2 * pyo.value(m.matriz[3])
        + 3 * pyo.value(m.matriz[4])
    )
    print(
        f"Total produzido: {int(round(medias))} médias e {int(round(grandes))} grandes"
    )


Custo Total Mínimo: 297.0

Matriz 1: 63 chapas
Matriz 4: 117 chapas
Total produzido: 504 médias e 351 grandes
